In [20]:
# Model EUR Aggressive - COMPLETE ANALYSIS
# Full notebook: ISIN, weight, risk/return analysis, andrecommendations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import yfinance as yf
from typing import List, Dict, Optional
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# ── Global constants ──────────────────────────────────────────
TRADING_DAYS = 252      # Trading days per year
DEFAULT_RF   = 0.02     # Annual risk-free rate (2%)

print("✓ Portfolio Engine Ready")
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

✓ Portfolio Engine Ready
Date: 2026-03-03 04:38


In [48]:
# Data Loading

def load_portfolio(path: str) -> pd.DataFrame:
    """
    Load CSV with columns: ISIN, Name, Allocation (%).
    Auto-detects column names, normalizes weights, drops zero-weight rows.
    Returns DataFrame with columns: isin, name, allocation_w (sum = 1.0)
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower()

    isin_col   = next((c for c in df.columns if "isin" in c), None)
    name_col   = next((c for c in df.columns if any(x in c for x in ["name","nome","security"])), None)
    weight_col = next((c for c in df.columns if any(x in c for x in ["alloc","peso","weight","%"])), None)

    if not all([isin_col, name_col, weight_col]):
        raise ValueError(f"Required columns not found. Available: {df.columns.tolist()}")

    df = df[[isin_col, name_col, weight_col]].copy()
    df.columns = ["isin", "name", "allocation_pct"]

    # Clean and convert weights
    df["allocation_pct"] = (
        pd.to_numeric(
            df["allocation_pct"].astype(str).str.replace("%", "").str.strip(),
            errors="coerce"
        )
    )
    df = df[df["allocation_pct"].notna() & (df["allocation_pct"] > 0)].copy()

    # Clean ISINs
    df["isin"] = df["isin"].astype(str).str.strip().str.upper()
    df = df[df["isin"].str.len() >= 5]

    # Normalize to sum = 1.0
    df["allocation_w"] = df["allocation_pct"] / df["allocation_pct"].sum()

    return df[["isin", "name", "allocation_w"]].reset_index(drop=True)


# ISIN → Yahoo Finance ticker mapping
# Extend this dict as needed for your assets
ISIN_TICKER_MAP: Dict[str, str] = {
    # ── ETF ────────────────────────────────────────────────────────────
    "FR0007054358": "MEUD.PA",     # Amundi EURO STOXX 50 II ETF
    "IE00B3ZW0K18": "IUSE.L",      # iShares S&P 500 EUR Hedged UCITS ETF (Acc)
    "IE00BD5CTX77": "SHYD.L",      # BNY Mellon Global Short-Dated HY Bond ETF
    "IE0009JOT9U1": "IGLN.L",      # iShares Physical Gold EUR Hedged ETC
    "IE000NVDQXE1": "DFND.L",      # First Trust Indxx Global Aerospace & Defence
    "IE000BN4N0T47": "IFRA.L",     # FTGT ClearBridge Global Infrastructure Income Fund  ← verify
    "IE000YYE6WK5B": "DFEN.L",     # VanEck Defense UCITS ETF
    "IE00BYXW4642": "CIE.L",       # CIM Dividend Income Fund C Ordinary Shares  ← verify
    "US37960A3703": "DRIV",        # Global X U.S. Electrification ETF (USD)
    "JE00B24DKH53B": "WGASP.L",   # WisdomTree Natural Gas 1x Daily Short EUR  ← verify
    "XS3188157393": "BRCN.L",      # 12M EUR BO INVERSE BRCN ON NVDA AND AMD  ← structured product
    "XS3188156668": "WPHX.L",      # 12M EUR WOM Phoenix AC on NVDA and AMD 9  ← structured product
    
    # ── Fondi attivi (Morningstar/Yahoo) ───────────────────────────────
    "LU0297942434": "0P0000XMSJ.L",  # BGF Global Corporate Bond A2 EUR H
    "LU0140363002": "0P00009QZM.L",  # Franklin Mutual European Fund A(acc) EUR
    "LU0248184466": "0P00006Q25.L",  # Schroder ISF Asian Opportunities EUR A
    "LU0106817157": "0P00006Q2A.L",  # Schroder SISF Emerging Europe EUR A
    "LU0360484769": "0P0000YWDH.L",  # Morgan Stanley Investment Funds - US Advantage
    "LU1775980201": "0P0001BTHF.L",  # Invesco UK Equity Fund E EUR
    "LU1650492330": "0P0001H7WB.L",  # Amundi FTSE 100 UCITS ETF EUR Hedged Acc
    "LU0359201455": "0P0000XMSL.L",  # BGF China Hedged A2 EUR
    "LU0512094193": "0P0000YQTD.L",  # Morgan Stanley Investment Funds - Japanese Equity
    "IE0003867441E": "0P0000YQKF.L", # BNY Mellon Small Cap Euroland Fund EUR A Acc

    "CASH-LIQ":     "CASH",
}

def map_isin_to_ticker(isin: str, name: str = "") -> str:
    """Map ISIN → Yahoo Finance ticker. Returns UNKNOWN_<prefix> if not found."""
    return ISIN_TICKER_MAP.get(isin, f"UNKNOWN_{isin[:8]}")


def download_prices(
    tickers: List[str],
    start: str = "2020-01-01",
    end: Optional[str] = None
) -> pd.DataFrame:
    """
    Download adjusted close prices from Yahoo Finance.
    CASH is represented as a flat line at 1.0.
    Returns a forward-filled DataFrame, dropping columns with >50% NaN.
    """
    valid_tickers = [t for t in tickers if t not in ("CASH",) and not t.startswith("UNKNOWN")]
    prices = pd.DataFrame()

    invalid_tickers = []

    # Scarica i dati solo per i tickers validi
    for ticker in valid_tickers:
        try:
            raw = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
            if 'Close' in raw:
                prices[ticker] = raw['Close']
            else:
                invalid_tickers.append(ticker)
        except Exception as e:
            print(f"⚠ Error downloading data for ticker {ticker}: {e}")
            invalid_tickers.append(ticker)

    # Aggiungi "CASH" come una colonna flat con valore 1.0
    if "CASH" in tickers:
        prices["CASH"] = 1.0

    # Rimuovi asset con più del 50% di dati mancanti
    threshold = 0.5
    prices = prices.loc[:, prices.isna().mean() < threshold]

    # Forward fill i dati
    prices = prices.ffill().dropna(how="all")

    # Verifica se ci sono tickers che non hanno dati
    if invalid_tickers:
        print(f"\n⚠ I seguenti tickers non hanno serie storiche valide: {', '.join(invalid_tickers)}")

    return prices


print("✓ Data loading functions defined")

✓ Data loading functions defined


In [49]:
# Risk Metrics

def annualized_return(rets: pd.Series) -> float:
    """Annualized arithmetic mean return."""
    return float(rets.mean() * TRADING_DAYS)

def annualized_volatility(rets: pd.Series) -> float:
    """Annualized standard deviation of returns."""
    return float(rets.std() * np.sqrt(TRADING_DAYS))

def sharpe_ratio(rets: pd.Series, rf: float = DEFAULT_RF) -> float:
    """Annualized Sharpe ratio."""
    vol = annualized_volatility(rets)
    if vol == 0:
        return np.nan
    excess = rets - rf  # Corretta gestione del tasso di rischio privo di rischio
    return float(annualized_return(excess) / vol)

def sortino_ratio(rets: pd.Series, rf: float = DEFAULT_RF) -> float:
    """Annualized Sortino ratio (penalizes only downside volatility)."""
    excess = rets - rf
    downside = excess[excess < 0]
    if len(downside) == 0:
        return np.inf
    downside_vol = float(downside.std() * np.sqrt(TRADING_DAYS))
    if downside_vol == 0:
        return np.inf
    return float(annualized_return(excess) / downside_vol)

def max_drawdown(rets: pd.Series) -> float:
    """Maximum peak-to-trough drawdown (negative value)."""
    cum  = (1 + rets).cumprod()
    peak = cum.cummax()
    dd   = (cum / peak) - 1
    return float(dd.min())

def calmar_ratio(rets: pd.Series) -> float:
    """Calmar ratio: annualized return / |max drawdown|."""
    mdd = max_drawdown(rets)
    if mdd == 0:
        return np.inf
    return float(annualized_return(rets) / abs(mdd))

def historical_var(rets: pd.Series, alpha: float = 0.95) -> float:
    """Historical VaR at confidence level alpha (daily, negative value)."""
    return float(np.quantile(rets, 1 - alpha))

def cvar(rets: pd.Series, alpha: float = 0.95) -> float:
    """Conditional VaR (Expected Shortfall) at confidence level alpha."""
    var = historical_var(rets, alpha)
    return float(rets[rets <= var].mean())

def compute_metrics(rets: pd.Series, name: str = "Portfolio") -> Dict:
    """Compute a full set of risk/return metrics for a return series."""
    return {
        "name":           name,
        "ann_return":     annualized_return(rets),
        "ann_volatility": annualized_volatility(rets),
        "sharpe":         sharpe_ratio(rets),
        "sortino":        sortino_ratio(rets),
        "max_drawdown":   max_drawdown(rets),
        "calmar":         calmar_ratio(rets),
        "var_95":         historical_var(rets, 0.95),
        "cvar_95":        cvar(rets, 0.95),
        "skew":           float(rets.skew()),
        "kurtosis":       float(rets.kurtosis()),
    }

print("✓ Risk metrics functions defined")

✓ Risk metrics functions defined


In [50]:
# Analytics and Visualization

def plot_allocation_pie(df: pd.DataFrame, title: str = "Portfolio Allocation"):
    """Pie chart of portfolio weights."""
    fig, ax = plt.subplots(figsize=(9, 6))
    df_plot = df.set_index("name")["allocation_w"]
    if df_plot.sum() == 0:
        print("⚠ No valid allocation data available to plot.")
        return
    df_plot.plot(kind="pie", autopct="%1.1f%%", ax=ax, startangle=90)
    ax.set_ylabel("")
    ax.set_title(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

def plot_correlation_heatmap(corr: pd.DataFrame, title: str = "Correlation Matrix"):
    """Annotated heatmap of asset correlations."""
    fig, ax = plt.subplots(figsize=(10, 8))
    if corr.empty:
        print("⚠ No correlation data available to plot.")
        return
    sns.heatmap(
        corr, annot=True, cmap="RdBu_r", center=0, fmt=".2f",
        square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax
    )
    ax.set_title(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

def plot_equity_curve(
    rets: pd.Series,
    bench_rets: Optional[pd.Series] = None,
    title: str = "Cumulative Returns"
):
    """Cumulative return chart, optionally vs. benchmark."""
    if rets.empty:
        print("⚠ No return data available to plot.")
        return
    fig, ax = plt.subplots(figsize=(12, 6))
    (1 + rets).cumprod().plot(ax=ax, label="Portfolio", linewidth=2)
    if bench_rets is not None:
        (1 + bench_rets).cumprod().plot(ax=ax, label="Benchmark", linewidth=2, linestyle="--")
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Cumulative Return")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_drawdown(rets: pd.Series, title: str = "Portfolio Drawdown"):
    """Drawdown chart with red shaded area."""
    if rets.empty:
        print("⚠ No return data available to plot drawdown.")
        return
    cum  = (1 + rets).cumprod()
    peak = cum.cummax()
    dd   = (cum / peak) - 1
    fig, ax = plt.subplots(figsize=(12, 5))
    dd.plot(ax=ax, color="red", linewidth=1.5)
    ax.fill_between(dd.index, dd.values, 0, color="red", alpha=0.3)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Drawdown")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_rolling_sharpe(rets: pd.Series, window: int = 63, title: str = "Rolling Sharpe Ratio (63d)"):
    """Rolling Sharpe ratio to monitor regime changes."""
    if rets.empty:
        print("⚠ No return data available to plot rolling Sharpe ratio.")
        return
    roll_mean = rets.rolling(window).mean() * TRADING_DAYS
    roll_std  = rets.rolling(window).std()  * np.sqrt(TRADING_DAYS)
    roll_sharpe = (roll_mean - DEFAULT_RF) / roll_std
    fig, ax = plt.subplots(figsize=(12, 4))
    roll_sharpe.plot(ax=ax, linewidth=1.5)
    ax.axhline(0, color="red",   linestyle="--", linewidth=1, alpha=0.7)
    ax.axhline(1, color="green", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Sharpe")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

def print_metrics_table(metrics_list: List[Dict]):
    """Pretty-print a comparison table of metrics."""
    df = pd.DataFrame(metrics_list).set_index("name").T
    fmt_map = {
        "ann_return":     "{:.2%}", "ann_volatility": "{:.2%}",
        "sharpe":         "{:.3f}", "sortino":        "{:.3f}",
        "max_drawdown":   "{:.2%}", "calmar":         "{:.3f}",
        "var_95":         "{:.2%}", "cvar_95":        "{:.2%}",
        "skew":           "{:.3f}", "kurtosis":       "{:.3f}",
    }
    print("\n" + "="*60)
    print("PERFORMANCE & RISK METRICS")
    print("="*60)
    for metric, row in df.iterrows():
        fmt = fmt_map.get(metric, "{:.4f}")
        # Ensure that infinite and NaN values are handled gracefully
        values = "  |  ".join(fmt.format(v) if np.isfinite(v) else "∞" for v in row)
        print(f"{metric:20s}: {values}")
    print("="*60 + "\n")

print("✓ Analytics & visualization functions defined")

✓ Analytics & visualization functions defined


In [67]:
# Loading Portfolio

import pandas as pd
from typing import Dict

# Lista degli ISIN e dei pesi (esempio)
portfolio_data = [
    ("IE00BD5CTX77", "BNY Mellon Global Short-Dated High Yield Bond", 5.00),
    ("LU0297942434", "BGF Global Corporate Bond Fund A2 EUR H", 7.50),
    ("XS3188157393", "12M EUR BO INVERSE BRCN ON NVDA AND AMD", 2.00),
    ("XS3188156668", "12M EUR WOM Phoenix AC on NVDA and AMD 9", 2.00),
    ("IE0003867441E", "BNY Mellon Small Cap Euroland Fund EUR A Acc", 5.00),
    ("IE00B3ZW0K18", "iShares S&P 500 EUR Hedged UCITS ETF (Acc)", 5.00),
    ("LU0360484769", "Morgan Stanley Investment Funds - US Advantage", 2.50),
    ("FR0007054358", "Amundi EURO STOXX 50 II UCITS ETF Acc", 7.50),
    ("LU1775980201", "Invesco UK Equity Fund E EUR", 5.00),
    ("LU0140363002", "Franklin Mutual European Fund A(acc) EUR", 10.00),
    ("LU1650492330", "Amundi FTSE 100 UCITS ETF EUR Hedged Acc", 2.50),
    ("LU0248184466", "Schroder ISF Asian Opportunities EUR A Acc", 7.50),
    ("LU0106817157", "Schroder SISF Emerging Europe Fund EUR A Acc", 7.50),
    ("IE00BYXW4642", "CIM Dividend Income Fund C Ordinary Shares", 2.50),
    ("LU0359201455", "BGF China Hedged A2 EUR", 2.50),
    ("LU0512094193", "Morgan Stanley Investment Funds - Japanese Equity", 2.50),
    ("IE0009JOT9U1", "iShares Physical Gold EUR Hedged ETC", 2.00),
    ("IE000NVDQXE1", "First Trust Indxx Global Aerospace & Defence", 2.50),
    ("IE000BN4N0T47", "FTGF ClearBridge Global Infrastructure Income Fund", 5.00),
    ("US37960A3703", "Global X U.S. Electrification ETF (USD)", 4.00),
    ("IE000YYE6WK5B", "VanEck Defense UCITS ETF EUR", 5.00),
    ("JE00B24DKH53B", "WisdomTree Natural Gas 1x Daily Short EUR", 0.00),
    ("Cash Equiv/Money Market", "-", 5.00),
]

# Mappatura ISIN → Ticker
ISIN_TICKER_MAP: Dict[str, str] = {
    "IE00BD5CTX77": "SHYD.L",  # BNY Mellon Global Short-Dated High Yield Bond ETF
    "LU0297942434": "0P0000XMSJ.L",  # BGF Global Corporate Bond Fund A2 EUR H
    "XS3188157393": "BRCN.L",  # 12M EUR BO INVERSE BRCN ON NVDA AND AMD
    "XS3188156668": "WPHX.L",  # 12M EUR WOM Phoenix AC on NVDA and AMD 9
    "IE0003867441E": "0P0000YQKF.L",  # BNY Mellon Small Cap Euroland Fund EUR A Acc
    "IE00B3ZW0K18": "IUSE.L",  # iShares S&P 500 EUR Hedged UCITS ETF (Acc)
    "LU0360484769": "0P0000YWDH.L",  # Morgan Stanley Investment Funds - US Advantage
    "FR0007054358": "MEUD.PA",  # Amundi EURO STOXX 50 II UCITS ETF Acc
    "LU1775980201": "0P0001BTHF.L",  # Invesco UK Equity Fund E EUR
    "LU0140363002": "0P00009QZM.L",  # Franklin Mutual European Fund A(acc) EUR
    "LU1650492330": "0P0001H7WB.L",  # Amundi FTSE 100 UCITS ETF EUR Hedged Acc
    "LU0248184466": "0P00006Q25.L",  # Schroder ISF Asian Opportunities EUR A Acc
    "LU0106817157": "0P00006Q2A.L",  # Schroder SISF Emerging Europe Fund EUR A Acc
    "IE00BYXW4642": "CIE.L",  # CIM Dividend Income Fund C Ordinary Shares
    "LU0359201455": "0P0000XMSL.L",  # BGF China Hedged A2 EUR
    "LU0512094193": "0P0000YQTD.L",  # Morgan Stanley Investment Funds - Japanese Equity
    "IE0009JOT9U1": "IGLN.L",  # iShares Physical Gold EUR Hedged ETC
    "IE000NVDQXE1": "DFND.L",  # First Trust Indxx Global Aerospace & Defence
    "IE000BN4N0T47": "IFRA.L",  # FTGF ClearBridge Global Infrastructure Income Fund
    "US37960A3703": "DRIV",  # Global X U.S. Electrification ETF (USD)
    "IE000YYE6WK5B": "DFEN.L",  # VanEck Defense UCITS ETF EUR
    "JE00B24DKH53B": "WGASP.L",  # WisdomTree Natural Gas 1x Daily Short EUR
    "CASH-LIQ": "CASH",  # Cash Equiv/Money Market (Non tradable asset)
}

# Crea un DataFrame dal portafoglio
df_portfolio = pd.DataFrame(portfolio_data, columns=["isin", "name", "allocation_pct"])

# Normalizza i pesi (devono sommare a 1)
df_portfolio["allocation_w"] = df_portfolio["allocation_pct"] / df_portfolio["allocation_pct"].sum()

print(f"✓ Portfolio loaded — {len(df_portfolio)} assets, total weight: {df_portfolio['allocation_w'].sum():.2%}")

# Mappatura ISIN → Ticker
df_portfolio["ticker"] = df_portfolio["isin"].map(ISIN_TICKER_MAP).fillna(
    df_portfolio["isin"].apply(lambda x: f"UNKNOWN_{x[:8]}")
)

# Verifica i ticker mappati
print("\n✓ Tickers mapped")
display(df_portfolio[["name", "ticker", "allocation_w"]])

# Avvisa sugli ISIN non mappati
unknown = df_portfolio[df_portfolio["ticker"].str.startswith("UNKNOWN")]
if not unknown.empty:
    print(f"\n⚠ {len(unknown)} unmapped tickers — add them to ISIN_TICKER_MAP:")
    # Salva gli ISIN non mappati in un file log per future modifiche
    unknown.to_csv("unmapped_tickers.csv", index=False)
    for _, row in unknown.iterrows():
        print(f"   ISIN: {row['isin']}  |  Name: {row['name']}")
else:
    print("\n✓ All tickers mapped successfully")

# Controlla se la somma dei pesi è corretta (con una piccola tolleranza)
tolerance = 1e-6  # Piccola tolleranza per errori di floating point
weight_sum = df_portfolio['allocation_w'].sum()
if abs(weight_sum - 1.0) > tolerance:
    print(f"⚠ Warning: Total weight is {weight_sum:.4f}. It should sum to 1.0.")
else:
    print(f"✓ Total weight is {weight_sum:.4f}, as expected.")

✓ Portfolio loaded — 23 assets, total weight: 100.00%

✓ Tickers mapped


,name,ticker,allocation_w
0,BNY Mellon Global Short-Dated High Yield Bond,SHYD.L,0.050
1,BGF Global Corporate Bond Fund A2 EUR H,0P0000XMSJ.L,0.075
2,12M EUR BO INVERSE BRCN ON NVDA AND AMD,BRCN.L,0.020
3,12M EUR WOM Phoenix AC on NVDA and AMD 9,WPHX.L,0.020
4,BNY Mellon Small Cap Euroland Fund EUR A Acc,0P0000YQKF.L,0.050
5,iShares S&P 500 EUR Hedged UCITS ETF (Acc),IUSE.L,0.050
6,Morgan Stanley Investment Funds - US Advantage,0P0000YWDH.L,0.025
7,Amundi EURO STOXX 50 II UCITS ETF Acc,MEUD.PA,0.075
8,Invesco UK Equity Fund E EUR,0P0001BTHF.L,0.050
9,Franklin Mutual European Fund A(acc) EUR,0P00009QZM.L,0.100



⚠ 1 unmapped tickers — add them to ISIN_TICKER_MAP:
   ISIN: Cash Equiv/Money Market  |  Name: -
✓ Total weight is 1.0000, as expected.


In [74]:
# Verifica che i prezzi siano stati scaricati

import yfinance as yf
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Lista degli asset (ISIN, Nome, Pesi)
portfolio_data = [
    ("IE00BD5CTX77", "BNY Mellon Global Short-Dated High Yield Bond", 5.00),
    ("LU0297942434", "BGF Global Corporate Bond Fund A2 EUR H", 7.50),
    ("XS3188157393", "12M EUR BO INVERSE BRCN ON NVDA AND AMD", 2.00),
    ("XS3188156668", "12M EUR WOM Phoenix AC on NVDA and AMD 9", 2.00),
    ("IE0003867441E", "BNY Mellon Small Cap Euroland Fund EUR A Acc", 5.00),
    ("IE00B3ZW0K18", "iShares S&P 500 EUR Hedged UCITS ETF (Acc)", 5.00),
    ("LU0360484769", "Morgan Stanley Investment Funds - US Advantage", 2.50),
    ("FR0007054358", "Amundi EURO STOXX 50 II UCITS ETF Acc", 7.50),
    ("LU1775980201", "Invesco UK Equity Fund E EUR", 5.00),
    ("LU0140363002", "Franklin Mutual European Fund A(acc) EUR", 10.00),
    ("LU1650492330", "Amundi FTSE 100 UCITS ETF EUR Hedged Acc", 2.50),
    ("LU0248184466", "Schroder ISF Asian Opportunities EUR A Acc", 7.50),
    ("LU0106817157", "Schroder SISF Emerging Europe Fund EUR A Acc", 7.50),
    ("IE00BYXW4642", "CIM Dividend Income Fund C Ordinary Shares", 2.50),
    ("LU0359201455", "BGF China Hedged A2 EUR", 2.50),
    ("LU0512094193", "Morgan Stanley Investment Funds - Japanese Equity", 2.50),
    ("IE0009JOT9U1", "iShares Physical Gold EUR Hedged ETC", 2.00),
    ("IE000NVDQXE1", "First Trust Indxx Global Aerospace & Defence", 2.50),
    ("IE000BN4N0T47", "FTGF ClearBridge Global Infrastructure Income Fund", 5.00),
    ("US37960A3703", "Global X U.S. Electrification ETF (USD)", 4.00),
    ("IE000YYE6WK5B", "VanEck Defense UCITS ETF EUR", 5.00),
    ("JE00B24DKH53B", "WisdomTree Natural Gas 1x Daily Short EUR", 0.00),
    ("Cash Equiv/Money Market", "-", 5.00),
]

# Funzione per scaricare i dati da Yahoo Finance (senza API key)
def download_yahoo_data(ticker: str, start_date: str, end_date: str):
    try:
        data = yf.download(ticker, start=start_date, end=end_date, progress=False)
        if data.empty:
            print(f"⚠ No data found for {ticker} on Yahoo Finance.")
            return None
        return data[['Close']]  # Restituisce solo la colonna "Close" (aggiustata)
    except Exception as e:
        print(f"⚠ Error downloading {ticker} data from Yahoo: {e}")
        return None

# Funzione per eseguire lo scraping dei dati da Investing.com
def scrape_investing_data(ticker: str):
    url = f"https://www.investing.com/equities/{ticker}-historical-data"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"⚠ Failed to fetch data for {ticker} from Investing.com.")
            return None
        soup = BeautifulSoup(response.text, 'html.parser')
        table = soup.find('table', {'class': 'historicalTbl'})
        if not table:
            print(f"⚠ No historical data table found for {ticker} on Investing.com.")
            return None
        rows = table.find_all('tr')[1:]  # Skip header row
        data = []
        for row in rows:
            cols = row.find_all('td')
            if len(cols) > 0:
                date = cols[0].text.strip()
                close = cols[4].text.strip().replace(',', '')
                data.append([date, float(close)])
        df = pd.DataFrame(data, columns=["Date", "Close"])
        df['Date'] = pd.to_datetime(df['Date'], format='%b %d, %Y')
        df.set_index('Date', inplace=True)
        return df
    except Exception as e:
        print(f"⚠ Error scraping data for {ticker} from Investing.com: {e}")
        return None

# Funzione per eseguire lo scraping dei dati da Morningstar
def scrape_morningstar_data(ticker: str):
    url = f"https://www.morningstar.com/stocks/xnas/{ticker}/quote"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"⚠ Failed to fetch data for {ticker} from Morningstar.")
            return None
        soup = BeautifulSoup(response.text, 'html.parser')
        table = soup.find('table', {'class': 'table table-condensed'})
        if not table:
            print(f"⚠ No historical data table found for {ticker} on Morningstar.")
            return None
        rows = table.find_all('tr')[1:]  # Skip header row
        data = []
        for row in rows:
            cols = row.find_all('td')
            if len(cols) > 0:
                date = cols[0].text.strip()
                close = cols[1].text.strip().replace(',', '')
                data.append([date, float(close)])
        df = pd.DataFrame(data, columns=["Date", "Close"])
        df['Date'] = pd.to_datetime(df['Date'], format='%b %d, %Y')
        df.set_index('Date', inplace=True)
        return df
    except Exception as e:
        print(f"⚠ Error scraping data for {ticker} from Morningstar: {e}")
        return None

# Funzione principale per scaricare i dati da tutte le fonti per tutti gli asset
def download_all_assets(portfolio_data, start_date: str, end_date: str):
    all_data = {}
    for isin, name, weight in portfolio_data:
        print(f"\n📥 Attempting to download data for {name} (ISIN: {isin})...")
        
        # Yahoo Finance
        data = download_yahoo_data(name, start_date, end_date)
        if data is None:
            # Se Yahoo Finance fallisce, prova con Investing.com
            data = scrape_investing_data(name)
            if data is None:
                # Se Investing.com fallisce, prova con Morningstar
                data = scrape_morningstar_data(name)
        
        if data is not None:
            all_data[name] = data
            print(f"✓ Data successfully downloaded for {name}.")
        else:
            print(f"⚠ No data found for {name}.")
    
    return all_data

# Esegui il download per tutti gli asset
start_date = "2021-01-01"
end_date = "2023-01-01"
all_assets_data = download_all_assets(portfolio_data, start_date, end_date)

# Mostra i risultati per i primi 5 asset
for ticker, data in list(all_assets_data.items())[:5]:
    print(f"\nData for {ticker}:")
    print(data.head())


📥 Attempting to download data for BNY Mellon Global Short-Dated High Yield Bond (ISIN: IE00BD5CTX77)...


$SHORT-DATED: possibly delisted; no timezone found
$YIELD: possibly delisted; no timezone found
$GLOBAL: possibly delisted; no timezone found
$MELLON: possibly delisted; no timezone found

4 Failed downloads:
['SHORT-DATED', 'YIELD', 'GLOBAL', 'MELLON']: possibly delisted; no timezone found
$GLOBAL: possibly delisted; no timezone found


✓ Data successfully downloaded for BNY Mellon Global Short-Dated High Yield Bond.

📥 Attempting to download data for BGF Global Corporate Bond Fund A2 EUR H (ISIN: LU0297942434)...


$A2: possibly delisted; no timezone found
$EUR: possibly delisted; no price data found  (1d 2021-01-01 -> 2023-01-01)
$BGF: possibly delisted; no price data found  (1d 2021-01-01 -> 2023-01-01)
$CORPORATE: possibly delisted; no timezone found

5 Failed downloads:
['GLOBAL', 'A2', 'CORPORATE']: possibly delisted; no timezone found
['EUR', 'BGF']: possibly delisted; no price data found  (1d 2021-01-01 -> 2023-01-01)
$EUR: possibly delisted; no price data found  (1d 2021-01-01 -> 2023-01-01)


✓ Data successfully downloaded for BGF Global Corporate Bond Fund A2 EUR H.

📥 Attempting to download data for 12M EUR BO INVERSE BRCN ON NVDA AND AMD (ISIN: XS3188157393)...


...

⚠ Failed to fetch data for VanEck Defense UCITS ETF EUR from Morningstar.
⚠ No data found for VanEck Defense UCITS ETF EUR.

📥 Attempting to download data for WisdomTree Natural Gas 1x Daily Short EUR (ISIN: JE00B24DKH53B)...


$DAILY: possibly delisted; no timezone found
$NATURAL: possibly delisted; no timezone found
$1X: possibly delisted; no timezone found
$WISDOMTREE: possibly delisted; no timezone found
$SHORT: possibly delisted; no timezone found
$GAS: possibly delisted; no price data found  (1d 2021-01-01 -> 2023-01-01)

7 Failed downloads:
['EUR', 'GAS']: possibly delisted; no price data found  (1d 2021-01-01 -> 2023-01-01)
['DAILY', 'NATURAL', '1X', 'WISDOMTREE', 'SHORT']: possibly delisted; no timezone found


⚠ No data found for WisdomTree Natural Gas 1x Daily Short EUR on Yahoo Finance.
⚠ Failed to fetch data for WisdomTree Natural Gas 1x Daily Short EUR from Investing.com.


$-: possibly delisted; no timezone found

1 Failed download:
['-']: possibly delisted; no timezone found


⚠ Failed to fetch data for WisdomTree Natural Gas 1x Daily Short EUR from Morningstar.
⚠ No data found for WisdomTree Natural Gas 1x Daily Short EUR.

📥 Attempting to download data for - (ISIN: Cash Equiv/Money Market)...
⚠ No data found for - on Yahoo Finance.
⚠ Failed to fetch data for - from Investing.com.
⚠ Failed to fetch data for - from Morningstar.
⚠ No data found for -.

Data for BNY Mellon Global Short-Dated High Yield Bond:
Price           Close                                                
Ticker            BNY       BOND GLOBAL HIGH MELLON SHORT-DATED YIELD
Date                                                                 
2021-01-04  11.188890  91.740570    NaN  NaN    NaN         NaN   NaN
2021-01-05  11.366613  91.626945    NaN  NaN    NaN         NaN   NaN
2021-01-06  11.258437  91.334801    NaN  NaN    NaN         NaN   NaN
2021-01-07  11.150258  91.091278    NaN  NaN    NaN         NaN   NaN
2021-01-08  11.088439  90.993874    NaN  NaN    NaN         NaN   NaN

D

In [76]:
# Data Manipulation

import pandas as pd

# Funzione per pulire i dati (rimuovere NaN, formattare correttamente)
def clean_data(all_assets_data):
    cleaned_data = {}
    
    for asset_name, data in all_assets_data.items():
        if data is not None and not data.empty:
            # Rimuovere righe con valori NaN
            data = data.dropna()
            
            # Assicurarsi che la colonna 'Date' sia come indice
            if data.index.name != 'Date':
                data.set_index('Date', inplace=True)
            
            # Aggiungere il DataFrame pulito al dizionario
            cleaned_data[asset_name] = data
        else:
            print(f"⚠ No valid data found for {asset_name}.")
    
    return cleaned_data

# Funzione per generare un report sui dataset scaricati
def report_downloaded_data(cleaned_data):
    print(f"\n✓ Report sui dati scaricati:")
    total_assets = len(cleaned_data)
    valid_assets = sum(1 for data in cleaned_data.values() if not data.empty)
    print(f"Total assets in portfolio: {total_assets}")
    print(f"Valid assets with data downloaded: {valid_assets}")
    
    # Mostrare i primi 3 dataset (se ci sono)
    for asset_name, data in list(cleaned_data.items())[:3]:
        print(f"\n{asset_name}:")
        print(data.head())
    
    # Restituisce il numero di asset validi
    return valid_assets, total_assets

# Funzione per unificare tutti i dati in un singolo DataFrame
def unify_data(cleaned_data):
    # Creare un DataFrame con tutti i dati
    all_prices = pd.DataFrame()
    
    for asset_name, data in cleaned_data.items():
        if data is not None and not data.empty:
            # Aggiungere i dati dei prezzi come una nuova colonna
            all_prices[asset_name] = data['Close']
    
    # Rimuovere righe con NaN (se ci sono)
    all_prices.dropna(inplace=True)
    
    return all_prices

# Pulizia dei dati scaricati
cleaned_data = clean_data(all_assets_data)

# Report sul numero di dataset scaricati e puliti
valid_assets, total_assets = report_downloaded_data(cleaned_data)

# Unificare tutti i dati in un DataFrame
all_prices = unify_data(cleaned_data)

# Mostrare il DataFrame finale
print(f"\n✓ Unified DataFrame with all asset prices (cleaned):")
print(all_prices.head())

# Ora all_prices contiene i prezzi storici di tutti gli asset con una data unica come indice


✓ Report sui dati scaricati:
Total assets in portfolio: 12
Valid assets with data downloaded: 0

BNY Mellon Global Short-Dated High Yield Bond:
Empty DataFrame
Columns: [(Close, BNY), (Close, BOND), (Close, GLOBAL), (Close, HIGH), (Close, MELLON), (Close, SHORT-DATED), (Close, YIELD)]
Index: []

BGF Global Corporate Bond Fund A2 EUR H:
Empty DataFrame
Columns: [(Close, A2), (Close, BGF), (Close, BOND), (Close, CORPORATE), (Close, EUR), (Close, FUND), (Close, GLOBAL), (Close, H)]
Index: []

12M EUR BO INVERSE BRCN ON NVDA AND AMD:
Empty DataFrame
Columns: [(Close, 12M), (Close, AMD), (Close, AND), (Close, BO), (Close, BRCN), (Close, EUR), (Close, INVERSE), (Close, NVDA), (Close, ON)]
Index: []

✓ Unified DataFrame with all asset prices (cleaned):
Empty DataFrame
Columns: []
Index: []


In [78]:
# ====================================
# COMPUTE RETURNS & PER-ASSET METRICS
# ====================================

# Calcolare i ritorni giornalieri per ciascun asset
returns = prices.pct_change().dropna()
print(f"✓ Returns computed: {returns.shape}")

# Calcolare le metriche per ciascun asset (saltando 'CASH')
asset_metrics = [
    compute_metrics(returns[t], name=t)
    for t in returns.columns if t != "CASH"
]

# Creare un DataFrame con le metriche per ciascun asset
df_asset_metrics = pd.DataFrame(asset_metrics)

# Unire con i nomi e i pesi del portafoglio
df_asset_metrics = df_asset_metrics.merge(
    df_portfolio[["ticker", "name", "allocation_w"]],
    left_on="name", right_on="ticker", how="left"
).drop(columns=["ticker"])

# Ordinare le metriche per Sharpe ratio in ordine decrescente
print("\n" + "="*80)
print("METRICS PER ASSET (sorted by Sharpe Ratio)")
print("="*80)

# Visualizzare TUTTI gli asset (senza limitarsi a 3)
# Rimuoviamo il limite di visualizzazione a 3 e mostriamo l'intero DataFrame
display(
    df_asset_metrics.sort_values("sharpe", ascending=False)[
        ["name_y", "allocation_w", "ann_return", "ann_volatility", "sharpe", "sortino", "max_drawdown"]
    ].rename(columns={"name_y": "asset_name"})
)

✓ Returns computed: (476, 3)

METRICS PER ASSET (sorted by Sharpe Ratio)


,asset_name,allocation_w,ann_return,ann_volatility,sharpe,sortino,max_drawdown
0,iShares S&P 500 EUR Hedged UCITS ETF (Acc),0.05,0.045197,0.189069,0.133268,-29.227984,-0.244639
1,iShares Physical Gold EUR Hedged ETC,0.02,-0.000290,0.146380,-0.138611,-36.879257,-0.179992
2,Global X U.S. Electrification ETF (USD),0.04,-0.022738,0.304664,-0.140278,-21.102197,-0.360881


In [79]:
# ====================================
# PORTFOLIO-LEVEL ANALYSIS
# ====================================

# Allineare i pesi alle colonne effettive nei ritorni
weights_s = (
    df_portfolio.set_index("ticker")["allocation_w"]
    .reindex(returns.columns)  # Allinea i pesi con i tickers effettivi
    .fillna(0)  # Se un asset non è presente nei ritorni, imposta il peso a 0
)

# Rinizializzare i pesi in modo che sommino a 1 (nel caso di asset mancanti)
weights_s /= weights_s.sum()

# Calcolare i ritorni del portafoglio come somma ponderata dei ritorni giornalieri
port_returns = (returns * weights_s.values).sum(axis=1)

# Calcolare le metriche del portafoglio
port_metrics = compute_metrics(port_returns, name="Portfolio")

# Stampare i risultati delle metriche del portafoglio
print("\n" + "="*60)
print("PORTFOLIO METRICS")
print("="*60)

# Mostrare le metriche con formattazione adeguata
for k, v in port_metrics.items():
    if k == "name":  # Non mostrare il nome
        continue
    fmt = "{:.2%}" if k in ("ann_return","ann_volatility","max_drawdown","var_95","cvar_95") else "{:.4f}"
    print(f"  {k:20s}: {fmt.format(v)}")

print("="*60)


PORTFOLIO METRICS
  ann_return          : 1.22%
  ann_volatility      : 18.15%
  sharpe              : -0.0429
  sortino             : -30.6968
  max_drawdown        : -24.71%
  calmar              : 0.0495
  var_95              : -1.94%
  cvar_95             : -2.45%
  skew                : -0.0527
  kurtosis            : 0.7474


In [81]:
import pandas as pd

# Lista degli ISIN e relativi pesi del portafoglio
portfolio_data = [
    ("IE00BD5CTX77", "BNY Mellon Global Short-Dated High Yield Bond", 5.00),
    ("LU0297942434", "BGF Global Corporate Bond Fund A2 EUR H", 7.50),
    ("XS3188157393", "12M EUR BO INVERSE BRCN ON NVDA AND AMD", 2.00),
    ("XS3188156668", "12M EUR WOM Phoenix AC on NVDA and AMD 9", 2.00),
    ("IE0003867441E", "BNY Mellon Small Cap Euroland Fund EUR A Acc", 5.00),
    ("IE00B3ZW0K18", "iShares S&P 500 EUR Hedged UCITS ETF (Acc)", 5.00),
    ("LU0360484769", "Morgan Stanley Investment Funds - US Advantage", 2.50),
    ("FR0007054358", "Amundi EURO STOXX 50 II UCITS ETF Acc", 7.50),
    ("LU1775980201", "Invesco UK Equity Fund E EUR", 5.00),
    ("LU0140363002", "Franklin Mutual European Fund A(acc) EUR", 10.00),
    ("LU1650492330", "Amundi FTSE 100 UCITS ETF EUR Hedged Acc", 2.50),
    ("LU0248184466", "Schroder ISF Asian Opportunities EUR A Acc", 7.50),
    ("LU0106817157", "Schroder SISF Emerging Europe Fund EUR A Acc", 7.50),
    ("IE00BYXW4642", "CIM Dividend Income Fund C Ordinary Shares", 2.50),
    ("LU0359201455", "BGF China Hedged A2 EUR", 2.50),
    ("LU0512094193", "Morgan Stanley Investment Funds - Japanese Equity", 2.50),
    ("IE0009JOT9U1", "iShares Physical Gold EUR Hedged ETC", 2.00),
    ("IE000NVDQXE1", "First Trust Indxx Global Aerospace & Defence", 2.50),
    ("IE000BN4N0T47", "FTGF ClearBridge Global Infrastructure Income Fund", 5.00),
    ("US37960A3703", "Global X U.S. Electrification ETF (USD)", 4.00),
    ("IE000YYE6WK5B", "VanEck Defense UCITS ETF EUR", 5.00),
    ("JE00B24DKH53B", "WisdomTree Natural Gas 1x Daily Short EUR", 0.00),
    ("Cash Equiv/Money Market", "-", 5.00),
]

# Carica il portafoglio in un DataFrame
df_portfolio = pd.DataFrame(portfolio_data, columns=["isin", "name", "allocation_pct"])

# Normalizza i pesi (devono sommare a 1)
df_portfolio["allocation_w"] = df_portfolio["allocation_pct"] / df_portfolio["allocation_pct"].sum()

# Mostra il portafoglio
print(f"✓ Portfolio loaded — {len(df_portfolio)} assets, total weight: {df_portfolio['allocation_w'].sum():.2%}")
print(df_portfolio)

✓ Portfolio loaded — 23 assets, total weight: 100.00%
                       isin  \
0              IE00BD5CTX77   
1              LU0297942434   
2              XS3188157393   
3              XS3188156668   
4             IE0003867441E   
5              IE00B3ZW0K18   
6              LU0360484769   
7              FR0007054358   
8              LU1775980201   
9              LU0140363002   
10             LU1650492330   
11             LU0248184466   
12             LU0106817157   
13             IE00BYXW4642   
14             LU0359201455   
15             LU0512094193   
16             IE0009JOT9U1   
17             IE000NVDQXE1   
18            IE000BN4N0T47   
19             US37960A3703   
20            IE000YYE6WK5B   
21            JE00B24DKH53B   
22  Cash Equiv/Money Market   

                                                 name  allocation_pct  \
0       BNY Mellon Global Short-Dated High Yield Bond             5.0   
1             BGF Global Corporate Bond Fund A2 EUR H  

In [ ]:
import yfinance as yf

# Mappa degli ISIN ai ticker di Yahoo Finance
ISIN_TICKER_MAP = {
    "IE00BD5CTX77": "SHYD.L",     # BNY Mellon Global Short-Dated HY Bond ETF
    "LU0297942434": "0P0000XMSJ.L",  # BGF Global Corporate Bond A2 EUR H
    "XS3188157393": "BRCN.L",    # 12M EUR BO INVERSE BRCN ON NVDA AND AMD
    "XS3188156668": "WPHX.L",    # 12M EUR WOM Phoenix AC on NVDA and AMD 9
    "IE0003867441E": "0P0000YQKF.L",  # BNY Mellon Small Cap Euroland Fund EUR A Acc
    # Aggiungere altri asset alla mappa se necessario...
}

def download_prices(tickers: list, start="2020-01-01", end="2023-01-01"):
    prices = pd.DataFrame()
    for ticker in tickers:
        try:
            data = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
            if 'Close' in data:
                prices[ticker] = data['Close']
        except Exception as e:
            print(f"⚠ Error downloading data for {ticker}: {e}")
    return prices

# Lista di ticker per il portafoglio
tickers = [ISIN_TICKER_MAP.get(isin, "UNKNOWN") for isin in df_portfolio['isin']]
prices = download_prices(tickers)

# Mostra i primi dati scaricati
print(f"✓ Data downloaded: {prices.shape[0]} rows × {prices.shape[1]} columns")
print(prices.head())